# Pipecat Smart Turn — Dataset Exploration

Explore the `pipecat-ai/smart-turn-data-v3.2` dataset before training.

- Label distribution (complete vs incomplete)
- Audio duration statistics
- Language / filler / synthetic breakdowns
- Listen to samples & visualise mel spectrograms

In [ ]:
from datasets import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display as ipd
from transformers import WhisperFeatureExtractor
from collections import Counter

## 1. Load Dataset

In [ ]:
ds_train = load_dataset("pipecat-ai/smart-turn-data-v3.2-train", split="train")
ds_test = load_dataset("pipecat-ai/smart-turn-data-v3.2-test", split="train")

print(f"Train: {len(ds_train):,} samples")
print(f"Test:  {len(ds_test):,} samples")
print(f"\nColumns: {ds_train.column_names}")
print(f"Features: {ds_train.features}")

In [ ]:
# Quick peek at a few rows
ds_train[:3]

## 2. Label Distribution

In [ ]:
labels = np.array(ds_train["endpoint_bool"])
complete = labels.sum()
incomplete = len(labels) - complete

print(f"Complete (1):   {complete:>7,}  ({complete/len(labels)*100:.1f}%)")
print(f"Incomplete (0): {incomplete:>7,}  ({incomplete/len(labels)*100:.1f}%)")
print(f"Pos weight (for BCE): {incomplete / max(complete, 1):.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Incomplete (0)", "Complete (1)"], [incomplete, complete])
ax.set_ylabel("Count")
ax.set_title("Label Distribution — Train")
plt.tight_layout()
plt.show()

## 3. Audio Duration Statistics

In [ ]:
# Sample a subset to compute durations (full scan can be slow on 270k samples)
N = min(10_000, len(ds_train))
rng = np.random.default_rng(42)
indices = rng.choice(len(ds_train), size=N, replace=False)

durations = []
sample_rates = set()
for i in indices:
    audio = ds_train[int(i)]["audio"]
    sr = audio["sampling_rate"]
    sample_rates.add(sr)
    durations.append(len(audio["array"]) / sr)

durations = np.array(durations)
print(f"Sampled {N:,} audio clips")
print(f"Sample rates seen: {sample_rates}")
print(f"Duration — min: {durations.min():.2f}s, max: {durations.max():.2f}s, "
      f"mean: {durations.mean():.2f}s, median: {np.median(durations):.2f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(durations, bins=80, edgecolor="black", linewidth=0.3)
ax.axvline(8.0, color="red", linestyle="--", label="8s model input cap")
ax.set_xlabel("Duration (s)")
ax.set_ylabel("Count")
ax.set_title(f"Audio Duration Distribution (n={N:,})")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Metadata Breakdowns

In [ ]:
# Build a metadata DataFrame (no audio, fast)
meta_cols = [c for c in ds_train.column_names if c != "audio"]
df = pd.DataFrame({c: ds_train[c] for c in meta_cols})
df.head()

In [ ]:
# Language breakdown
lang_counts = df["language"].value_counts()
print("Language distribution:")
print(lang_counts.to_string())

fig, ax = plt.subplots(figsize=(8, 3))
lang_counts.plot.bar(ax=ax)
ax.set_ylabel("Count")
ax.set_title("Samples per Language")
plt.tight_layout()
plt.show()

In [ ]:
# Filler type breakdown
df["filler_type"] = "nofiller"
df.loc[df["midfiller"] == True, "filler_type"] = "midfiller"
df.loc[df["endfiller"] == True, "filler_type"] = "endfiller"

filler_label = pd.crosstab(df["filler_type"], df["endpoint_bool"])
filler_label.columns = ["Incomplete", "Complete"]
print(filler_label)

filler_label.plot.bar(figsize=(6, 3), title="Filler Type × Label")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Synthetic vs real
synth_label = pd.crosstab(df["synthetic"], df["endpoint_bool"])
synth_label.columns = ["Incomplete", "Complete"]
synth_label.index = ["Real", "Synthetic"]
print(synth_label)

synth_label.plot.bar(figsize=(5, 3), title="Synthetic × Label")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Source dataset breakdown
print("Source dataset counts:")
print(df["dataset"].value_counts().to_string())

## 5. Listen to Samples

In [ ]:
def play_sample(ds, idx):
    """Display an audio player and metadata for a dataset sample."""
    sample = ds[idx]
    audio = sample["audio"]
    sr = audio["sampling_rate"]
    arr = np.array(audio["array"], dtype=np.float32)
    dur = len(arr) / sr
    label = "Complete" if sample["endpoint_bool"] else "Incomplete"
    print(f"[{idx}] {label} | lang={sample['language']} | "
          f"midfiller={sample['midfiller']} endfiller={sample['endfiller']} | "
          f"synthetic={sample['synthetic']} | {dur:.2f}s | dataset={sample['dataset']}")
    return ipd.Audio(arr, rate=sr)

In [ ]:
# A few complete samples
complete_idx = [i for i, v in enumerate(ds_train["endpoint_bool"][:5000]) if v]
for idx in complete_idx[:3]:
    display(play_sample(ds_train, idx))

In [ ]:
# A few incomplete samples
incomplete_idx = [i for i, v in enumerate(ds_train["endpoint_bool"][:5000]) if not v]
for idx in incomplete_idx[:3]:
    display(play_sample(ds_train, idx))

## 6. Mel Spectrogram Visualisation

In [ ]:
extractor = WhisperFeatureExtractor(chunk_length=8)

def plot_mel(ds, idx, ax=None):
    """Plot the 80×800 mel spectrogram for a sample."""
    sample = ds[idx]
    audio = sample["audio"]
    features = extractor(
        audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="np"
    )
    mel = features["input_features"][0]  # (80, 800)
    label = "Complete" if sample["endpoint_bool"] else "Incomplete"

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3))
    ax.imshow(mel, aspect="auto", origin="lower", cmap="inferno")
    ax.set_xlabel("Time frame")
    ax.set_ylabel("Mel bin")
    ax.set_title(f"[{idx}] {label} — {sample['language']}")
    return mel

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for ax, idx in zip(axes[0], complete_idx[:2]):
    plot_mel(ds_train, idx, ax=ax)
for ax, idx in zip(axes[1], incomplete_idx[:2]):
    plot_mel(ds_train, idx, ax=ax)
plt.tight_layout()
plt.show()

## 7. Test Set Overview

In [ ]:
test_labels = np.array(ds_test["endpoint_bool"])
print(f"Test set: {len(ds_test):,} samples")
print(f"  Complete:   {test_labels.sum():,} ({test_labels.mean()*100:.1f}%)")
print(f"  Incomplete: {(~test_labels.astype(bool)).sum():,} ({(1-test_labels.mean())*100:.1f}%)")

test_meta = [c for c in ds_test.column_names if c != "audio"]
df_test = pd.DataFrame({c: ds_test[c] for c in test_meta})
print(f"\nLanguages: {df_test['language'].value_counts().to_dict()}")

---

Done! Use these findings to decide on any data filtering, augmentation, or rebalancing before launching training.